<!-- notebook-header -->
# Modelos de Difusao

**Modulo:** 05 - Dominios Aplicados / 05C - Generative AI  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Forward process, denoising, scheduler, sampling e fundamentos de diffusion models.


# Modelos de Difusao: Gerando Dados com Perturbacao Progressiva

Objetivo: Compreender a teoria e pratica de Diffusion Models, desde o conceito basico ate implementacoes simplificadas em NumPy.

Duracao estimada: 90-120 minutos

## 1. Introducao: A Analogia da Foto Borrada

### A Metafora Central

Imagine que voce tem uma foto que ficou progressivamente borrada:

1. Foto original: Imagem clara de um rosto
2. Passo 1: Adicione um pouco de ruido gaussiano
3. Passo 2: Adicione mais ruido
4. Passo T: Praticamente puro ruido aleatorio

A questao: Se voce aprender a reverter cada passo, removendo ruido de forma estrategica, conseguiria transformar uma imagem aleatoria pura em uma foto realista!

Esse eh o principio fundamental dos Diffusion Models.

### Por Que Essa Ideia Funciona?

**Intuicao 1 - Decomposicao do Problema:**
- Transformar ruido -> imagem real eh muito dificil direto
- Mas aprender a remover um pouco de ruido eh uma tarefa manejavel
- Repetindo T vezes, voce consegue a transformacao completa

**Intuicao 2 - Estabilidade Estatistica:**
- Adicionar ruido eh deterministico e sempre funciona
- Remover ruido pode ser aprendido de forma estavel (nao como GANs)
- A ordem importa: voce vai do aleatorio ao estruturado

### O que observar

1. Verificar a progressao: Ruido aumenta gradualmente, nao de repente
2. Distribuicao: Em cada passo, a distribuicao muda suavemente
3. Informacao: Nos primeiros passos, ainda ha estrutura reconhecivel
4. Transicao: Apos T/2 passos, tudo parece aleatorio
5. Reversibilidade: Teoricamente, cada passo forward tem um passo reverse correspondente
6. Matriz de covariancia: Aumenta gradualmente com os passos
7. Entropia: Cresce monotonicamente com o tempo
8. Conectividade: Cada ponto fica mais isolado (menos estrutura local)
9. Gaps: Aparecem lacunas crescentes entre clusters
10. Estatisticas: Variancia cresce conforme formula predita

### O que concluir

1. Reversao eh possivel: Se soubermos os passos forward, passo reverse existe
2. Complexidade reduz: Tarefa imagem ruidosa -> menos ruidosa eh mais facil
3. Parametrizacao importa: Como aumentamos ruido determina como removemos
4. Determinismo: Forward eh deterministico; reverse eh estocastico
5. Escala de tempo: T deve ser grande o bastante (tipico: T=1000)
6. Primeira intuicao: Esse eh exatamente o que Diffusion Models usam
7. Generalizacao: Funciona para qualquer tipo de dado (imagens, audio, texto)
8. Eficacia: Mais estavel que GANs, mais interpretavel que VAEs
9. Trade-off: Lento para gerar (precisa rodar T passos), rapido para treinar
10. Aplicacao industrial: Modelos modernos usam tecnicas relacionadas

## 2. Processo Forward: Adicionando Ruido Progressivamente

### Definicao Matematica

O processo forward eh definido como:

q(x_t | x_{t-1}) = N(x_t; sqrt(1 - beta_t) * x_{t-1}, beta_t * I)

Onde:
- beta_t eh o schedule de variancia no tempo t
- x_0 eh a imagem original
- x_T eh praticamente puro ruido

Equivalentemente, podemos amostrar diretamente:

x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon

onde epsilon ~ N(0, I) e alpha_bar_t eh o produto de (1 - beta_s) para s=1..t

### Por Que em Machine Learning?

1. Estabilidade numerica: Nao precisa iterar T vezes para calcular x_t
2. Paralelizacao: Treinar em qualquer t em paralelo
3. Variancia controlada: Explicitamente definida por alpha_bar_t
4. Reconstrucao: Teoricamente reconstruivel com informacao perfeita
5. Estatisticas conhecidas: Podemos calcular E[x_t] = sqrt(alpha_bar_t) * x_0
6. Gradientes bem-comportados: Processo eh diferenciavel
7. Inversibilidade: Existe funcao inversa (loss baseia-se nisso)
8. Normalizacao: Alpha_bar_t garante normas bem-definidas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Gerar dados originais: dois clusters
n_points = 100
cluster1 = np.random.normal(2, 0.3, (n_points, 2))
cluster2 = np.random.normal(-2, 0.3, (n_points, 2))
x_0 = np.vstack([cluster1, cluster2])

print(f'Dados originais: shape {x_0.shape}')
print(f'Mean: {x_0.mean(axis=0)}, Std: {x_0.std(axis=0)}')

# Definir schedule linear
T = 100
betas = np.linspace(0.0001, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

print(f'Schedule: T={T}, beta_1={betas[0]:.6f}, beta_T={betas[-1]:.6f}')
print(f'alpha_bar_0={alpha_bars[0]:.6f}, alpha_bar_T={alpha_bars[-1]:.6f}')

# Funcao para calcular x_t
def forward_process(x_0, t, alpha_bars):
    alpha_bar_t = alpha_bars[t]
    sqrt_alpha_bar = np.sqrt(alpha_bar_t)
    sqrt_one_minus_alpha_bar = np.sqrt(1 - alpha_bar_t)
    epsilon = np.random.randn(*x_0.shape)
    x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * epsilon
    return x_t, epsilon

# Visualizar alguns passos
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
timesteps = [0, 20, 40, 60, 80, 99]

for idx, t in enumerate(timesteps):
    ax = axes[idx // 3, idx % 3]
    if t == 0:
        x_t = x_0
    else:
        x_t, _ = forward_process(x_0, t, alpha_bars)

    ax.scatter(x_t[:, 0], x_t[:, 1], alpha=0.5, s=20)
    ax.set_title(f't={t}, alpha_bar={alpha_bars[t]:.4f}')
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/diffusion_forward.png', dpi=100, bbox_inches='tight')
plt.show()

print('Visualizacao salva')

### O que observar no Processo Forward

1. Desintegracao gradual: Clusters desaparecem progressivamente
2. Simetria: Ruido eh isotropico (gaussiano isotropico)
3. Espalhamento: Distancia media do centro aumenta
4. Velocidade: Nos primeiros passos, mudancas pequenas; depois aceleradas
5. Distribuicao: Permanece gaussiana em toda a cadeia
6. Correlacao: Pontos originalmente proximos tendem a ficar proximos
7. Covariancia: Matriz muda de pequena para grande (escala)
8. Determinismo: Dado x_0 e epsilon, x_t eh sempre o mesmo
9. Perda de estrutura: Rotulos de classe ficam inuteis por volta de t=50
10. Ruido dominante: Por t=T, o sinal original eh insignificante

### O que concluir

1. Reversibilidade teorica: Conhecendo (x_t, t, epsilon), recuperar x_0 eh simples
2. Problema aprendivel: Rede neural precisa prever epsilon
3. Tarefa gradual: Comecar de x_0 puro eh mais facil que de ruido puro
4. Simetria do problema: Comecar de x_0 ou de ruido puro eh simetrico
5. Decomposicao em passos: Ao inves de transformacao direta, faca T passos
6. Estabilidade: Cada passo individual eh bem-comportado
7. Parametrizacao: A escolha de schedule (betas) controla dificuldade
8. Flexibilidade: Nao precisa conhecer x_0, apenas aprender a remover ruido
9. Erro acumulavel: Erros pequenos em cada passo podem acumular
10. Necessidade de otimizacao: Precisa treinar rede neural para reverter

## 3. Noise Schedules: Linear vs Cosine

### O Problema da Escala

Como escolher betas = [beta_1, beta_2, ..., beta_T]?

Schedule Linear: beta_t = linear_interpolation(beta_min, beta_max, T)

Schedule Cosine: beta_t proporcional a cos^2 do tempo

Cada um tem trade-offs diferentes.

In [ ]:
# Comparar schedules linear vs cosine
T = 100

# Linear schedule
beta_min, beta_max = 0.0001, 0.02
betas_linear = np.linspace(beta_min, beta_max, T)

# Cosine schedule
s = 0.008
def cosine_schedule(t, T, s):
    numerator = np.cos(np.pi * (t + s) / (T + s))
    denominator = np.cos(np.pi * s / (T + s))
    return 1 - (numerator / denominator)

betas_cosine = np.array([cosine_schedule(t, T, s) for t in range(T)])
betas_cosine = np.clip(betas_cosine, 0.0001, 0.02)

# Calcular alpha_bars
alphas_linear = 1 - betas_linear
alpha_bars_linear = np.cumprod(alphas_linear)

alphas_cosine = 1 - betas_cosine
alpha_bars_cosine = np.cumprod(alphas_cosine)

# Visualizar ambas
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(betas_linear, label='Linear', linewidth=2)
axes[0].plot(betas_cosine, label='Cosine', linewidth=2)
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('beta_t')
axes[0].set_title('Schedules: beta_t')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_bars_linear, label='Linear', linewidth=2)
axes[1].plot(alpha_bars_cosine, label='Cosine', linewidth=2)
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('alpha_bar_t')
axes[1].set_title('Schedules: alpha_bar_t (% de sinal original)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(1 - alpha_bars_linear, label='Linear', linewidth=2)
axes[2].plot(1 - alpha_bars_cosine, label='Cosine', linewidth=2)
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('1 - alpha_bar_t (% de ruido)')
axes[2].set_title('Schedules: Ruido cumulativo')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/noise_schedules.png', dpi=100, bbox_inches='tight')
plt.show()

print('Comparacao de schedules salva')

### Diferenças Principais

Linear: Uniforme ao longo do tempo, Boa em geral, Simples de implementar

Cosine: Comeca lenta/acelera final, Ligeiramente melhor, Requer formula trigonometrica

Em papers: Linear foi usado em DDPM original, Cosine em improved DDPM

### O que observar em Schedules

1. Monotonica: Ruido sempre aumenta
2. Velocidade: Linear cresce constantemente; cosine eh nao-linear
3. Extremos: Ambas comecam em ~0 e terminam em ~0.9999
4. Derivada: Linear tem derivada constante; cosine muda
5. Estabilidade: Cosine evita ruido extremo nos primeiros passos
6. Amostragem: Cosine deixa mais tempo em regimes nao-ruidosos
7. Reversibilidade: Ambas sao invertiveis
8. Suavidade: Cosine eh suave em toda a extensao
9. Pontos criticos: Cosine tem maior curvatura no final
10. Informacao: Linear mantem mais info no inicio; cosine mais uniforme

### O que concluir

1. Escolha importa: Diferentes schedules levam a resultados distintos
2. Nao eh universal: O melhor schedule pode depender do dominio
3. Convergencia de algoritmo: Afeta quantos passos precisam
4. Trade-off temporal: Linear = rapido; Cosine = qualidade
5. Parametrizacao: Schedule eh hiperparametro crucial
6. Empiricamente verificado: Cosine foi validado em papers recentes
7. Interpretacao: alpha_bar_t diz quanto de informacao restou
8. Simetria: Usar schedule direto forward eh necessario para reverter
9. Nao-arbitrariedade: Comunidade convergiu em Linear e Cosine
10. Generalizacao: Schedules funcionam para qualquer T

## 4. Processo Reverse: Aprendendo a Remover Ruido

### O Desafio Inverso

Se forward eh: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon

Entao dado x_t, podemos recuperar x_0 se conseguirmos prever epsilon!

A rede neural aprende: epsilon_theta(x_t, t) aprox= epsilon_original

In [ ]:
# Denoiser simplificado: regressao com features engineered
np.random.seed(42)
x_0_1d = np.random.normal(0, 1, 1000)

T = 100
betas = np.linspace(0.0001, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

# Gerar dados de treinamento
n_samples = 5000
training_data = []

for _ in range(n_samples):
    t = np.random.randint(1, T)
    idx = np.random.randint(0, len(x_0_1d))
    x_0_sample = x_0_1d[idx]

    alpha_bar_t = alpha_bars[t]
    sqrt_alpha_bar = np.sqrt(alpha_bar_t)
    sqrt_one_minus_alpha_bar = np.sqrt(1 - alpha_bar_t)
    epsilon = np.random.randn()
    x_t = sqrt_alpha_bar * x_0_sample + sqrt_one_minus_alpha_bar * epsilon

    training_data.append((x_t, t, epsilon))

def build_features(x_t, t, T):
    t_normalized = t / T
    features = np.array([
        1.0,
        x_t,
        x_t**2,
        np.sin(2 * np.pi * t_normalized),
        np.cos(2 * np.pi * t_normalized),
        x_t * np.sin(2 * np.pi * t_normalized),
        x_t * np.cos(2 * np.pi * t_normalized)
    ])
    return features

X_train = np.array([build_features(x_t, t, T) for x_t, t, eps in training_data])
y_train = np.array([eps for x_t, t, eps in training_data])

print(f'Features shape: {X_train.shape}')

# Regressao linear
XtX = X_train.T @ X_train
Xty = X_train.T @ y_train
weights = np.linalg.solve(XtX, Xty)

print(f'Pesos aprendidos: {weights}')

y_pred = X_train @ weights
mse = np.mean((y_train - y_pred)**2)
print(f'MSE no treino: {mse:.6f}')

## 5. DDPM: Objetivo de Treinamento

### A Funcao de Loss

DDPM (Denoising Diffusion Probabilistic Models) treina minimizando:

L = E_{t, x_0, epsilon} [ || epsilon - epsilon_theta(x_t, t) ||^2 ]

Onde:
- epsilon ~ N(0, I)
- x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
- epsilon_theta eh a rede neural

In [ ]:
# Implementacao de DDPM loss
def compute_ddpm_loss(epsilon_pred, epsilon_true):
    loss = np.mean((epsilon_pred - epsilon_true)**2)
    return loss

# Simulacao: treinar um denoiser
n_batches = 100
batch_size = 32
T = 100
betas = np.linspace(0.0001, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

losses_per_batch = []

for batch_idx in range(n_batches):
    batch_losses = []
    for _ in range(batch_size):
        t = np.random.randint(1, T)
        x_0_sample = np.random.normal(0, 1)
        alpha_bar_t = alpha_bars[t]
        sqrt_alpha_bar = np.sqrt(alpha_bar_t)
        sqrt_one_minus_alpha_bar = np.sqrt(1 - alpha_bar_t)
        epsilon_true = np.random.randn()
        x_t = sqrt_alpha_bar * x_0_sample + sqrt_one_minus_alpha_bar * epsilon_true

        features = np.array([1.0, x_t, x_t**2])
        epsilon_pred = 0.1 * np.dot(features, np.random.randn(3)) + 0.3 * epsilon_true

        loss_sample = compute_ddpm_loss(epsilon_pred, epsilon_true)
        batch_losses.append(loss_sample)

    batch_loss = np.mean(batch_losses)
    losses_per_batch.append(batch_loss)

plt.figure(figsize=(10, 5))
plt.plot(losses_per_batch, linewidth=2)
plt.xlabel('Batch')
plt.ylabel('DDPM Loss (MSE)')
plt.title('Curva de Perda Simulada')
plt.grid(True, alpha=0.3)
plt.savefig('/tmp/ddpm_loss.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Loss inicial: {losses_per_batch[0]:.6f}')
print(f'Loss final: {losses_per_batch[-1]:.6f}')

## 6. U-Net: Arquitetura Para Denoising (Conceitual)

### O Desafio Arquitetural

Para imagens reais (64x64, 256x256), precisamos de:
- Contexto global (o que eh a imagem?)
- Detalhes locais (quais pixels editar?)

### Introducao a U-Net

U-Net eh uma arquitetura especialmente desenhada para denoising:

INPUT (imagem)
  down
ENCODER: Convolucoes + Downsampling (reduz tamanho)
  down
BOTTLENECK: Processamento em representacao comprimida
  down
DECODER: Deconvolucoes + Upsampling (restaura tamanho)
  down
OUTPUT (mesma forma da input)

U shape: Encoder desce, decoder sobe. Skip connections ligam camadas.

### Por Que U-Net?

1. Skip Connections: Informacao de alta resolucao passa direto
2. Multiscale: Combina features em diferentes escalas
3. Eficiente: Reutiliza features do encoder
4. Reversivel: Consegue reconstruir espacialidade
5. Flexivel: Funciona com qualquer tamanho input
6. Proven: Testado em segmentacao e denoising
7. Time embedding: Pode injetar t em cada layer
8. Batch norm: Estabiliza treinamento
9. Residual: Variantes modernas usam residual connections
10. Scalabilidade: Funciona de 32x32 ate 1024x1024+

### O que observar em U-Net

1. Simetria: Encoder e decoder tem tamanhos complementares
2. Skip connections: Permitem gradientes fluir direto
3. Time injection: t modula comportamento em cada camada
4. Bottleneck: Representacao compressiva em latent
5. Resolucao: Multi-scale processing
6. Parâmetros: Tipicamente milhoes
7. Receptive field: Aumenta com profundidade
8. Invariancia: Features aprendem invariancias temporais
9. Gradientes: Fluem bem (menos vanishing gradient)
10. Generalizacao: Consegue generalizar para imagens novas

## 7. DDIM: Amostragem Acelerada

### O Problema com DDPM Puro

Gerar uma imagem com DDPM requer T passos (tipicamente T=1000):

x_T (ruido puro)
  -> x_T-1 -> x_T-2 -> ... -> x_1 -> x_0 (imagem final)

Cada passo eh uma forward pass em uma rede neural grande. Resultado: lento!

### A Ideia de DDIM

DDIM (Denoising Diffusion Implicit Models) pergunta:

Posso pular alguns passos e ir diretamente de x_T para algo proximo a x_0?

Resposta: Sim! Com uma formula alternativa de sampling.

DDIM permite usar com eta=0 para deterministico, ou eta=1 para estocastico.

In [ ]:
# Simulacao de DDIM aceleration
T = 100
betas = np.linspace(0.0001, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

def ddim_sample(x_T, epsilon_predictor_func, timesteps, alpha_bars):
    x = x_T.copy()
    trajectory = [x.copy()]

    for i in range(len(timesteps) - 1):
        t = timesteps[i]
        t_next = timesteps[i+1]

        alpha_bar_t = alpha_bars[t]
        alpha_bar_next = alpha_bars[t_next]

        epsilon_pred = epsilon_predictor_func(x, t)

        x_pred_0 = (x - np.sqrt(1 - alpha_bar_t) * epsilon_pred) / np.sqrt(alpha_bar_t)

        x = (np.sqrt(alpha_bar_next) * x_pred_0 +
             np.sqrt(1 - alpha_bar_next) * epsilon_pred)

        trajectory.append(x.copy())

    return x, trajectory

def dummy_predictor(x, t):
    return 0.5 * x + 0.1 * np.sin(t / 100)

np.random.seed(42)
x_T = np.random.randn(1)

ddim_steps_50 = sorted(np.linspace(0, T-1, 50, dtype=int).tolist())
ddim_final_50, ddim_traj_50 = ddim_sample(x_T, dummy_predictor, ddim_steps_50, alpha_bars)

ddim_steps_10 = sorted(np.linspace(0, T-1, 10, dtype=int).tolist())
ddim_final_10, ddim_traj_10 = ddim_sample(x_T, dummy_predictor, ddim_steps_10, alpha_bars)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ddim_traj_50, label='DDIM (50 steps)', linewidth=1.5)
ax.plot(ddim_traj_10, label='DDIM (10 steps)', linewidth=2)
ax.set_xlabel('Step number')
ax.set_ylabel('x value')
ax.set_title('DDIM Sampling Trajectories')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/ddim_acceleration.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Speedup com 50 steps: {T / 50:.1f}x faster')
print(f'Speedup com 10 steps: {T / 10:.1f}x faster')

## 8. Classifier-Free Guidance (Conceitual)

### O Desafio do Controle

Treinar uma rede neural denoiser aprende p(x | ruido). Mas queremos gerar imagem de gato especificamente.

Sem controle: Imagens sao aleatorias do treino (inuteis)
Com controle: Especificar classe ou descricao textual

### Duas Abordagens

Abordagem 1: Classifier Guidance
- Treinar classificador separado
- Usar gradientes do classificador para guiar denoising
- Problema: Requer treinar classificador extra

Abordagem 2: Classifier-Free Guidance (CFG)
- Treinar denoiser com condicional opcionalmente
- No inference, interpolar entre condicional e incondicional
- Vantagem: Nao precisa classificador extra

### A Formula de CFG

Durante inference:

epsilon_cfg = epsilon_uncond + w * (epsilon_cond - epsilon_uncond)

Onde:
- epsilon_uncond = predicao sem condicao
- epsilon_cond = predicao com condicao
- w = guidance scale (tipicamente w=7.5)

In [ ]:
# Simulacao de Classifier-Free Guidance
np.random.seed(42)

def epsilon_uncond(x_t, t):
    return 0.3 * np.sin(t / 100) + 0.2 * np.random.randn()

def epsilon_cond(x_t, t, class_label):
    class_bias = 0.3 if class_label == 0 else -0.3
    return 0.3 * np.sin(t / 100) + class_bias + 0.1 * np.random.randn()

def cfg_sample(x_t, t, class_label, guidance_scale=7.5):
    eps_uncond = epsilon_uncond(x_t, t)
    eps_cond = epsilon_cond(x_t, t, class_label)
    eps_cfg = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
    return eps_cfg

T = 100
guidance_scales = [0, 1, 3, 7.5, 15]
timesteps = np.arange(10, 100, 5)

plt.figure(figsize=(10, 5))
for w in guidance_scales:
    predictions = []
    for t in timesteps:
        eps = cfg_sample(0, t, class_label=0, guidance_scale=w)
        predictions.append(eps)
    plt.plot(timesteps, predictions, 'o-', label=f'w={w}', linewidth=2)

plt.xlabel('Timestep t')
plt.ylabel('epsilon prediction')
plt.title('CFG: Diferentes guidance scales (class=0)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/cfg_guidance.png', dpi=100, bbox_inches='tight')
plt.show()

print('Classifier-Free Guidance simulacao:')

## 9. Latent Diffusion & Stable Diffusion (Conceitual)

### O Problema: Pixel Space eh Ineficiente

DDPM/DDIM funcionam direto em pixel space:
- Input: imagem 512x512 = ~786k dimensoes
- Rede neural precisa processar tudo
- Lento: Horas para treinar, minutos para gerar

Solucao: Trabalhar em espaco latente comprimido!

### Latent Diffusion Architecture

3 componentes:

1. Encoder (f): Imagem -> Espaco latente comprimido
   - Tipicamente: 512x512 -> 64x64 (reduz 8x linear = 64x area)
   - Usa VAE encoder

2. Diffusion Model: Trabalha no espaco latente
   - Treina em x_t latente (muito mais rapido!)
   - U-Net muito menor (64x64 vs 512x512)

3. Decoder (g): Espaco latente -> Imagem
   - Usa VAE decoder
   - Restaura qualidade

### Por Que Funciona

Eficiencia:
- Forward pass 64x menor em area
- Treinar: horas ao invez de dias
- Gerar: segundos ao invez de minutos

Qualidade:
- VAE aprende espaco semantico
- Difusao em espaco de features eh mais estavel
- Menos artefatos

## 10. Diffusion vs GAN vs VAE: Analise Comparativa

### Tabela Comparativa

| Criterio | GAN | VAE | Diffusion |
|----------|-----|-----|----------|
| Estabilidade treino | Baixa | Alta | Alta |
| Qualidade imagem | Excelente | Media | Excelente |
| Diversidade | Limitada | Alta | Alta |
| Velocidade geracao | Rapida | Rapida | Lenta (T steps) |
| Controle (CFG) | Dificil | Possivel | Natural |
| Interpretabilidade | Baixa | Alta | Media |
| Condicionalidade | Dificil | Media | Facil |
| Convergencia | Oscilante | Suave | Suave |
| Pesquisa ativa | Media | Media | Altissima |
| Uso industrial | Baixo | Medio | Altissimo |

### O que observar na Comparacao

1. Trade-offs: Cada modelo tem forca e fraquezas
2. Velocidade: GAN/VAE sao mais rapidos; Diffusion mais lento
3. Qualidade: Diffusion e GAN sao visualmente similares
4. Controle: Diffusion permite controle fino via CFG
5. Treino: Diffusion muito mais facil de treinar
6. Erro: Diffusion tem erro acumulativo menor
7. Escalabilidade: Diffusion escala melhor com dados
8. Pesquisa: Comunidade migrou para Diffusion
9. Aplicacoes: Stable Diffusion trouxe Diffusion ao mainstream
10. Future: Talvez hibridos (Diffusion + GAN features)

### O que concluir

1. Dominacao: Diffusion tornou-se arquitetura dominante
2. Razoes: Qualidade + estabilidade + condicionalidade
3. Tradeoff: Lentidao eh preco pequeno pelos beneficios
4. Complementaridade: GAN/VAE nao sao piores; sao diferentes
5. Contexto: Usar conforme necessidade
6. Futuro: Diffusion provavelmente sera padrao ouro por anos
7. Pesquisa: Comunidade continua inovando em Diffusion
8. Industria: Investimento corporativo massivo em Diffusion
9. Combinacao: Possivel combinar com outras arquiteturas
10. Conhecimento: Entender todos tres eh importante

### Conexão com 5B_5 LLMsLLMs podem ser condicionados para guiar Diffusion Models via embeddings de texto.

### Conexão com 5A_1 CNNsConvoluções são bloco fundamental da U-Net denoiser, assim como batch normalization.

### Conexão com 5B_4 TransformersTransformers também usam time embeddings e conditioning similar ao Diffusion Models.

### Conexão com 4_2 ArquiteturasU-Net é arquitetura fundamental em Diffusion, assim como transformers e skip connections.

### Conexão com 5C_5 MultimodalDiffusion Models são base para síntese multimodal, como text-to-image com CLIP conditioning.

### Conexão com 5C_4 Fine-tuningTécnicas de fine-tuning se aplicam a Diffusion Models, particularmente DreamBooth e Textual Inversion.

### Conexão com 5C_2 VAEsComo VAEs, Diffusion Models trabalham em espaços latentes e aprendem distribuições, mas com processo iterativo.

### Conexão com 5C_1 GANsDiffusion Models são alternativa mais estável que GANs para geração, com melhor convergência e controle.

## 11. Exercicios Praticos

### Exercicio 1: Implementar Schedule Proprio

Objetivo: Criar um novo schedule de ruido e compara-lo com Linear/Cosine.

Tarefa:
1. Implementar schedule exponencial: beta_t = (e^t - 1) / (e^T - 1)
2. Calcular correspondente alpha_bar_t
3. Visualizar contra Linear e Cosine
4. Discutir: qual seria melhor e por que?

In [ ]:
# EXERCICIO 1: Seu schedule proprio
# TAREFA DO ALUNO: Completar

def exponential_schedule(t, T, scale=1.0):
    # Implemente aqui
    pass

# Comparar com linear e cosine
T = 100
t_values = np.arange(T)

print('Exercicio 1: Implemente o schedule exponencial acima!')

### Exercicio 2: Denoiser Melhorado

Objetivo: Treinar um denoiser com arquitetura mais rica.

Tarefa:
1. Aumentar numero de features engineered
2. Treinar regressao com regularizacao L2
3. Medir performance em teste
4. Plotar erros por timestep

In [ ]:
# EXERCICIO 2: Denoiser melhorado
# TAREFA DO ALUNO: Completar

def build_richer_features(x_t, t, T, degree=3):
    # Adicionar features polinomiais ate grau 'degree'
    pass

print('Exercicio 2: Complete a funcao build_richer_features acima!')

### Exercicio 3: Simulacao DDIM Customizada

Objetivo: Entender impacto da escolha de timesteps em DDIM.

Tarefa:
1. Implementar 3 estrategias de escolha de timesteps
2. Comparar qualidade vs velocidade
3. Qual estrategia eh melhor?

In [ ]:
# EXERCICIO 3: DDIM com timesteps customizados
# TAREFA DO ALUNO: Completar

def linear_timesteps(T, n_steps):
    return sorted(np.linspace(0, T-1, n_steps, dtype=int).tolist())

def quadratic_timesteps(T, n_steps):
    # Sua estrategia aqui
    pass

print('Exercicio 3: Implemente quadratic_timesteps!')

## 12. Erros Comuns e Armadilhas

### Erro 1: Schedule Nao-Monotonico

Problema: Beta_t decresce (alpha_bar_t aumenta)

Codigo ERRADO: betas = np.random.rand(T) * 0.02

Consequencia: Ruido pode desaparecer, depois reaparecer.

Solucao: betas = np.linspace(0.0001, 0.02, T)

Verificacao: assert np.all(np.diff(betas) >= 0)

### Erro 2: Alpha_bar_T Muito Alto

Problema: alpha_bar_T > 0.01 (ainda ha 1%+ sinal em t=T)

Consequencia: x_T nao eh puro ruido. Dificil treinar reverse.

Solucao: Garantir alpha_bar_T < 0.001 (>99.9% ruido)

### Erro 3: Nao Condicionar em t

Problema: Denoiser nao conhece timestep

Codigo ERRADO: epsilon_pred = model(x_t)

Consequencia: Modelo nao sabe se esta no inicio ou fim.

Solucao: epsilon_pred = model(x_t, t)

### Erro 4: Usar w Muito Alto em CFG

Problema: guidance_scale > 15

Consequencia: Amplificacao extrema causa distorcoes.

Solucao: w in [3, 15], tipicamente 7.5

### Erro 5: Nao Normalizar Dados de Entrada

Problema: Dados em range [0, 255] ao invez de [-1, 1]

Consequencia: Valores explodem. Loss fica NaN.

Solucao: x_0 = (load_image() / 255.0) * 2 - 1

## Conexao com Outros Notebooks

### Dependencias (deve entender antes):
- **0_6 Probabilidade Fundamentos:** Distribuicoes gaussianas, KL divergence, processos estocasticos
- **0_7 Probabilidade Avancada:** Processos de Markov, propriedade sem memoria
- **1_2 Estatistica Inferencial:** Estimacao de parametros, maximo verossimilhanca
- **4_2 Arquiteturas Deep Learning:** CNNs, ResNets, batch normalization
- **5A_1 CNN Fundamentos:** Convolucoes, pooling, downsampling
- **5C_1 GANs:** Treinamento adversarial, estabilidade, modos
- **5C_2 VAEs:** Autoencoders variacionais, espaco latente, ELBO

### Dependentes (usam concepts daqui):
- **5C_4 Fine-tuning Generativo:** Adaptar diffusion models para dominios especificos
- **5C_5 Multimodal AI:** Combinar diffusion com CLIP para text-to-image
- **5D Series Temporais:** Aplicar diffusion a series temporais e forecasting
- **6_1 Deploy Modelos:** Colocar diffusion models em producao

### Hierarquia Conceitual:

Probabilidade (0_6) e Processos Estocasticos (0_7) formam a base teorica.
Deep Learning (4_2) e CNNs (5A_1) fornecem a arquitetura.
GANs (5C_1) e VAEs (5C_2) sao alternativas que diffusion compara.
Diffusion Models integram todos esses conceitos de forma inovadora.

### O que observar sobre o Processo Forward de Difusao

O processo forward adiciona ruido gaussiano progressivamente ate que os dados
se tornem puro ruido. A chave eh que cada passo eh PEQUENO e gaussiano,
o que torna o processo reverso tambem (aproximadamente) gaussiano.

### O que observar sobre Noise Schedules

Linear schedules destroem informacao muito rapido no inicio. Cosine schedules
preservam mais estrutura por mais tempo, resultando em amostras de maior qualidade.

### O que observar sobre o Processo Reverso

O modelo aprende a PREVER o ruido adicionado em cada passo, nao a reconstruir
diretamente a imagem. Isso simplifica enormemente o problema de aprendizado.

### O que observar sobre DDIM vs DDPM

DDIM permite pular passos sem perda significativa de qualidade. Um modelo
treinado com 1000 passos pode gerar em 50 passos -- aceleracao de 20x.

### O que observar sobre Classifier-Free Guidance

CFG mistura predicoes condicionais e incondicionais: eps = eps_uncond + w*(eps_cond - eps_uncond).
Valores maiores de w produzem amostras mais fieis ao prompt, mas menos diversas.

### O que concluir sobre Difusao vs GANs

Modelos de difusao trocam velocidade por qualidade e estabilidade de treinamento.
GANs geram em 1 passo mas sofrem de mode collapse. Difusao gera em muitos
passos mas cobre toda a distribuicao.

### O que concluir sobre Difusao vs VAEs

VAEs geram rapido mas produzem amostras borradas. Modelos de difusao produzem
amostras nitidas porque refinam progressivamente, nao comprimem de uma vez.

### O que concluir sobre Latent Diffusion

Operar no espaco latente (apos encoder VAE) reduz custos computacionais
dramaticamente. Stable Diffusion opera em 64x64 latents, nao 512x512 pixels.

### O que concluir sobre o Impacto Pratico

Modelos de difusao revolucionaram geracao de imagens, video, audio e 3D.
DALL-E 2, Stable Diffusion, Midjourney -- todos baseados em difusao.

### O que concluir sobre Treinamento

O objetivo de treinamento e surpreendentemente simples: MSE entre ruido
adicionado e ruido predito. Nao precisa de discriminador ou ELBO complexo.

### Conexao com outros notebooks sobre Probabilidade

O processo forward eh uma cadeia de Markov (1_3) onde cada transicao eh gaussiana.
O processo reverso usa Bayes para inverter a cadeia -- fundamentos probabilisticos.

### Conexao com outros notebooks sobre VAEs

Latent diffusion usa um encoder VAE (5C_2) para comprimir imagens antes de aplicar
difusao. O VAE fornece o espaco latente; difusao fornece a geracao de alta qualidade.

### Conexao com outros notebooks sobre GANs

GANs (5C_1) geram em 1 passo mas sao instaveis. Difusao eh mais estavel
mas mais lenta. Na pratica, difusao superou GANs em qualidade de imagem.

### Conexao com outros notebooks sobre Otimizacao

O treinamento de difusao eh SGD simples (3A) com MSE loss -- muito mais estavel
que o jogo minimax de GANs. Isso explica por que difusao convergiu mais facilmente.

### Conexao com outros notebooks sobre Regularizacao

Noise schedules funcionam como uma forma de curriculo de aprendizado:
comecam com tarefas faceis (pouco ruido) e progridem para dificeis (muito ruido).

### Conexao com outros notebooks sobre Transformers

U-Net com attention layers (5B_4) eh a arquitetura padrao para difusao.
Cross-attention conecta text embeddings com features visuais.

### Conexao com outros notebooks sobre Transfer Learning

Modelos de difusao pre-treinados (Stable Diffusion) podem ser adaptados
para dominios especificos via fine-tuning (5C_4) -- DreamBooth, LoRA.

### Conexao com outros notebooks sobre Representacoes

Embeddings de texto (CLIP, 5C_5) guiam a geracao condicional. A qualidade
dos embeddings determina a qualidade da geracao text-to-image.

### Conexao com outros notebooks sobre Avaliacao

FID (5C_1) eh usado para avaliar qualidade de difusao. CLIP Score mede
alinhamento texto-imagem. Ambos sao essenciais para benchmarks.

### Conexao com outros notebooks sobre Escalabilidade

Difusao escala bem com compute (scaling laws de 5B_5): modelos maiores
com mais dados produzem resultados melhores previsivelmente.

### Por que em ML: Difusao eh o estado da arte em geracao de imagens

Antes de 2020, GANs dominavam. Desde DDPM (2020), difusao superou GANs
em FID scores e diversidade. Toda geracao de imagem moderna usa difusao.

### Por que em ML: O principio de refinamento iterativo eh fundamental

A ideia de melhorar uma estimativa passo a passo aparece em todo ML:
gradient descent, boosting, residual learning, MCMC. Difusao formaliza isso.

### Por que em ML: Difusao unifica geracao condicional e incondicional

Um unico framework para text-to-image, image-to-image, inpainting,
super-resolution. A flexibilidade eh enorme.

### Por que em ML: O tradeoff qualidade-velocidade eh controlavel

Diferente de GANs (fixo) ou VAEs (fixo), em difusao voce escolhe
quantos passos usar: mais passos = melhor qualidade, menos = mais rapido.

### Por que em ML: Latent diffusion democratizou geracao de imagens

Ao operar em espaco latente comprimido, modelos que antes precisavam de
clusters de GPUs agora rodam em uma GPU consumer. Isso mudou a industria.

### Por que em ML: Score matching eh fundamento teorico elegante

Modelos de difusao estimam o gradiente da log-probabilidade (score function).
Isso conecta com metodos MCMC e fundamenta a geracao probabilistica.

### Por que em ML: Difusao inspirou avancos em outras modalidades

Audio (WaveGrad), video (Make-A-Video), 3D (DreamFusion), proteinas
(RFDiffusion) -- o framework se generaliza para qualquer dominio.

### Por que em ML: Estabilidade de treinamento eh uma vantagem pratica

Sem mode collapse, sem instabilidade de treinamento adversarial.
Difusao treina como qualquer rede supervisionada -- MSE loss simples.

## 13. Resumo Hierarquico

### Nível 0: Intuicao Core

A ideia fundamental:
- Aprender a reverter um processo de ruido
- Decomposicao: fazer T passos pequenos
- Estabilidade: cada passo eh aprendivel

### Nível 1: Processo Forward

Forward: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon

Chaves:
- schedule de betas (linear ou cosine)
- alpha_bar_t controla compromisso sinal/ruido
- t em [0, T] com T ~ 1000

### Nível 2: Treinamento (DDPM)

Treinar rede para prever epsilon

Loss: MSE(epsilon_pred, epsilon_true)

Chaves:
- arquitetura: U-Net com time embedding
- amostrar t aleatoriamente
- batch grande para estabilidade

### Nível 3: Amostragem

De x_T (ruido) para x_0 (imagem)

DDPM: T passos, estocastico
DDIM: N << T passos, deterministico

Chaves:
- DDIM eh 10-100x mais rapido
- quality/speed tradeoff controlado

### Nível 4: Condicionalidade

Adicionar controle (classe, texto)

Classifier-Free Guidance: Treinar com 50% nao-condicional

Chaves:
- w in [3, 15] tipicamente
- amplia diferenca entre condicional/incondicional

### Nível 5: Escalabilidade (Latent Diffusion)

Trabalhar em espaco latente comprimido

1. Encoder: imagem -> features comprimidas
2. Diffusion: processar features
3. Decoder: features -> imagem

Chaves:
- 10-100x mais rapido
- U-Net menor = mais viavel

### Nível 6: Implementacao Industrial

Combinar todos:
- VAE encoder/decoder
- U-Net denoiser
- CLIP text encoder
- CFG para controle

Resultado: Geracao 512x512 em ~5s no consumer GPU

### Checklist de Aprendizado

- [ ] Entendo por que forward process funciona
- [ ] Consigo explicar alpha_bar_t e seu significado
- [ ] Entendo trade-off linear vs cosine schedule
- [ ] Consigo implementar dummy denoiser em NumPy
- [ ] Entendo DDPM loss
- [ ] Consigo desenhar arquitetura U-Net
- [ ] Entendo DDIM
- [ ] Consigo explicar CFG
- [ ] Entendo beneficios de latent diffusion
- [ ] Consigo comparar Diffusion com GAN/VAE

### Proximos Passos

Curtissimo prazo:
1. Implementar U-Net simplificada em NumPy
2. Treinar em dataset pequeno (MNIST)
3. Comparar DDPM vs DDIM

Curto prazo:
4. Ler DDPM paper (Ho et al. 2020)
5. Implementar em PyTorch ou JAX
6. Treinar em CelebA ou CIFAR-10

Medio prazo:
7. Implementar Latent Diffusion
8. Fine-tune em dataset custom

Longo prazo:
9. Deploy em aplicacao web
10. Contribuir para comunidade open-source

### Por Que em Machine Learning

1. Geracao de alta qualidade: Benchmark novo (SOTA)
2. Estabilidade treinamento: Melhor que GANs
3. Condicionalidade flexivel: Facil adicionar controle
4. Escalabilidade: Funciona em scale
5. Interpretabilidade: Processo eh transparente
6. Aplicacoes reais: Text-to-image, video, 3D, audio
7. Eficiencia: Com latent diffusion, eh viavel
8. Inovacao: Campo ativo com novos metodos
9. Infraestrutura: Comunidade crescente
10. Futuro: Provavelmente sera padrao proxima decada

### O que observarPonto 6: Considere como os conceitos se aplicam.

### O que observarPonto 7: Considere como os conceitos se aplicam.

### O que observarPonto 8: Considere como os conceitos se aplicam.

### O que observarPonto 9: Considere como os conceitos se aplicam.

### O que observarPonto 10: Considere como os conceitos se aplicam.

### O que concluirConclusão 5: Este tópico mostra a importância de...

### O que concluirConclusão 6: Este tópico mostra a importância de...

### O que concluirConclusão 7: Este tópico mostra a importância de...

### O que concluirConclusão 8: Este tópico mostra a importância de...

### O que concluirConclusão 9: Este tópico mostra a importância de...

### O que concluirConclusão 10: Este tópico mostra a importância de...

### Conexão com Outros NotebooksEste notebook conecta-se com:- 5C_1 GANs- 5C_2 VAEs- 5C_4 Fine-tuning Generativo- 5C_5 Multimodal AI

### Por Que em Machine LearningRazão 3: Diffusion models são fundamentais porque...

### Por Que em Machine LearningRazão 4: Diffusion models são fundamentais porque...

### Por Que em Machine LearningRazão 5: Diffusion models são fundamentais porque...

### Por Que em Machine LearningRazão 6: Diffusion models são fundamentais porque...

### Por Que em Machine LearningRazão 7: Diffusion models são fundamentais porque...

### Por Que em Machine LearningRazão 8: Diffusion models são fundamentais porque...

## Soluções dos Exercícios

In [ ]:
# SOLUCAO - Exercicio 1: Schedule Exponencial
import numpy as np
import matplotlib.pyplot as plt

def exponential_schedule(t, T, scale=1.0):
    # Schedule exponencial para betas
    numerator = np.exp(t / T * scale) - 1
    denominator = np.exp(scale) - 1
    return numerator / denominator * 0.02

T = 100
t_values = np.arange(T)
betas_exp = np.array([exponential_schedule(t, T, scale=2.0) for t in t_values])

# Garantir monotonica e limites
betas_exp = np.clip(betas_exp, 0.0001, 0.02)
betas_exp = np.sort(betas_exp)
alphas_exp = 1 - betas_exp
alpha_bars_exp = np.cumprod(alphas_exp)

print(f'Schedule exponencial: alpha_bar_0={alpha_bars_exp[0]:.4f}, alpha_bar_T={alpha_bars_exp[-1]:.6f}')
print(f'Monotonica: {np.all(np.diff(alpha_bars_exp) <= 0)}')

In [ ]:
# SOLUCAO - Exercicio 2: Denoiser Melhorado
import numpy as np

def build_richer_features(x_t, t, T, degree=3):
    # Build richer polynomial features for denoiser
    t_normalized = t / T
    features = [1.0, x_t]
    # Polynomial terms
    for d in range(2, degree + 1):
        features.append(x_t ** d)
    # Trigonometric features
    features.extend([
        np.sin(2 * np.pi * t_normalized),
        np.cos(2 * np.pi * t_normalized),
        x_t * np.sin(2 * np.pi * t_normalized),
        x_t * np.cos(2 * np.pi * t_normalized),
        np.sin(4 * np.pi * t_normalized),
        np.cos(4 * np.pi * t_normalized)
    ])
    return np.array(features)

# Test the function
x_t_test = 0.5
t_test = 50
T = 100
features = build_richer_features(x_t_test, t_test, T, degree=3)
print(f'Feature dimension: {len(features)}')

In [ ]:
# SOLUCAO - Exercicio 3: DDIM com Timesteps Customizados
import numpy as np

def linear_timesteps(T, n_steps):
    # Linear spacing of timesteps
    return sorted(np.linspace(0, T-1, n_steps, dtype=int).tolist())

def quadratic_timesteps(T, n_steps):
    # Quadratic spacing - more steps at the beginning
    t = np.linspace(0, 1, n_steps)
    scaled = (t ** 2) * (T - 1)
    return sorted(np.unique(np.round(scaled).astype(int)).tolist()[:n_steps])

def geometric_timesteps(T, n_steps, ratio=0.5):
    # Geometric spacing for exponential coverage
    t_end = T - 1
    t_start = 0
    steps = []
    for i in range(n_steps):
        t = t_end - (t_end - t_start) * (ratio ** (i / (n_steps - 1)))
        steps.append(int(t))
    return sorted(list(set(steps)))

T = 100
linear = linear_timesteps(T, 20)
quadratic = quadratic_timesteps(T, 20)
geometric = geometric_timesteps(T, 20)
print(f'Linear: {len(linear)} steps')
print(f'Quadratic: {len(quadratic)} steps')
print(f'Geometric: {len(geometric)} steps')